<a href="https://colab.research.google.com/github/derekhatherley/ds2002-fa26/blob/main/2026_09_25_%E2%80%94_Cleaning_Gauntlet_%E2%80%94_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Cleaning Gauntlet

**Lab — 2026-09-25 · Fall 2026**  

---

## Lab 05 — Cleaning Gauntlet

Three hundred rows, generated messy. This is the first dataset in the course you cannot eyeball, which means you have to work from counts and assertions rather than from looking at the table and deciding it seems fine.

Deliverables: a clean frame, a decision log, a set of assertions that pass, and one business number at the end — revenue by category — that you would be willing to defend.

Keep the log as you go. Reconstructing it afterward is much harder than writing one line per step, and the write-up at the end depends on it.

### The log

Run this first, then call `log(...)` after each cleaning step.

In [192]:
DECISIONS = []

def log(step, decision, rows_affected):
    DECISIONS.append({'step': step, 'decision': decision, 'rows': rows_affected})
    print(f'[{step}] {decision} ({rows_affected} row(s))')

In [193]:
import pandas as pd, numpy as np
from io import StringIO
rng = np.random.default_rng(5)
items = ['Cheeseburger','cheese burger','Foam Finger','foam finger','Rain Poncho','rain poncho']
cats = ['Food','food','Merch','Apparel','RainGear','rain-gear']
rows = []
for i in range(300):
    rows.append({
        'order_id': i,
        'item': rng.choice(items),
        'category': rng.choice(cats),
        'qty': rng.choice([1,2,3,-1,np.nan], p=[.5,.25,.15,.05,.05]),
        'price': rng.choice(['$7.50','7.5','$12.00','24','6.0']),
    })
df = pd.DataFrame(rows)
df = pd.concat([df, df.sample(15, random_state=1)])  # inject dupes
df.head()

,order_id,item,category,qty,price
0,0,Rain Poncho,RainGear,3.0,$12.00
1,1,foam finger,Apparel,1.0,7.5
2,2,cheese burger,Merch,1.0,$7.50
3,3,Cheeseburger,Food,NaN,$7.50
4,4,cheese burger,Apparel,1.0,7.5


### TODO 1 — drop duplicates

In [194]:
# TODO
original_rows = len(df)
rawdf = df.copy()
df = df.drop_duplicates().reset_index(drop = True)
new_rows = len(df)
affected_rows = original_rows - new_rows
log('Drop duplicates', 'Copied original data and removed duplicate rows', affected_rows)
df.head()

[Drop duplicates] Copied original data and removed duplicate rows (15 row(s))


,order_id,item,category,qty,price
0,0,Rain Poncho,RainGear,3.0,$12.00
1,1,foam finger,Apparel,1.0,7.5
2,2,cheese burger,Merch,1.0,$7.50
3,3,Cheeseburger,Food,NaN,$7.50
4,4,cheese burger,Apparel,1.0,7.5


### TODO 2 — clean `price` -> float

In [195]:
# TODO

affected_rows = df['price'].astype(str).str.contains('$').sum()
df['price'] = df['price'].astype(str)
df['price'] = df['price'].str.replace('$', '')
df['price'] = df['price'].astype(float)
log('Clean price', 'Removed all dollar signs and changed data type to float', affected_rows)
df.head()

rev1 = (df['price']*df['qty']).sum()
print('The revenue at this stage is: $', rev1)

[Clean price] Removed all dollar signs and changed data type to float (300 row(s))
The revenue at this stage is: $ 4594.5


### TODO 3 — `qty` -> numeric, drop rows with missing/negative qty

In [196]:
# TODO

affected_rows = df['qty'].isna().sum() + (df['qty'] < 0).sum()
df = df.dropna()
df = df[df['qty'] > 0]
df['qty'] = df['qty'].astype(int)
log('Clean qty', 'Removed NaN or negative quantity values', affected_rows)
df.head()

rev2 = (df['price']*df['qty']).sum()
print('The revenue at this stage is: $', rev2)

[Clean qty] Removed NaN or negative quantity values (25 row(s))
The revenue at this stage is: $ 4740.0


### TODO 4 — canonicalize `item`

Six spellings, three real products. Start by listing what you actually have, then build the mapping from that list rather than from memory.

```python
print(df['item'].value_counts())
ITEM_MAP = {...}
```

In [197]:
# TODO: inspect the variants, build a mapping dict, apply it, log the collapse


print(df['item'].value_counts())
ITEM_MAP = {
    'Cheeseburger': 'Cheeseburger',
    'cheese burger': 'Cheeseburger',
    'Foam Finger': 'Foam Finger',
    'foam finger': 'Foam Finger',
    'Rain Poncho': 'Rain Poncho',
    'rain poncho': 'Rain Poncho'
}

starting_items = df['item'].copy()
df['item'] = df['item'].map(ITEM_MAP)
changed = (starting_items != df['item']).sum()

log('Canonicalize item', 'Changed all lowercase instances to form with first letter capitalized, logging how many rows changed', changed)

df.head()


item
Foam Finger      57
Rain Poncho      49
cheese burger    44
Cheeseburger     43
rain poncho      42
foam finger      40
Name: count, dtype: int64
[Canonicalize item] Changed all lowercase instances to form with first letter capitalized, logging how many rows changed (126 row(s))


,order_id,item,category,qty,price
0,0,Rain Poncho,RainGear,3,12.0
1,1,Foam Finger,Apparel,1,7.5
2,2,Cheeseburger,Merch,1,7.5
4,4,Cheeseburger,Apparel,1,7.5
5,5,Foam Finger,food,3,6.0


### TODO 5 — normalize `category`

Same approach. Note that `Apparel` and `Merch` are a business decision, not a string problem — decide and log it.

In [198]:
# TODO

print(df['category'].value_counts())
CATEGORY_MAP = {
    'Food': 'Food',
    'food': 'Food',
    'Merch': 'Merch', # Treating Merch and Apparel as synonoymous
    'Apparel': 'Merch',
    'RainGear': 'RainGear',
    'rain-gear': 'RainGear'
}

starting_categories = df['category'].copy()
df['category'] = df['category'].map(CATEGORY_MAP)
changed = (starting_categories != df['category']).sum()

log('Normalize category', 'Mapped each category to one of 3 categories, treating apparel as a synonym of merch', changed)

df.head()

category
Food         51
Merch        51
rain-gear    45
food         44
Apparel      43
RainGear     41
Name: count, dtype: int64
[Normalize category] Mapped each category to one of 3 categories, treating apparel as a synonym of merch (132 row(s))


,order_id,item,category,qty,price
0,0,Rain Poncho,RainGear,3,12.0
1,1,Foam Finger,Merch,1,7.5
2,2,Cheeseburger,Merch,1,7.5
4,4,Cheeseburger,Merch,1,7.5
5,5,Foam Finger,Food,3,6.0


### TODO 6 — prove it's clean

**TODO:** uncomment these and add two more assertions of your own — one about the item names and one about the categories.

In [199]:
assert df.duplicated().sum() == 0
assert df['qty'].min() >= 1
assert df['price'].dtype == float
# TODO: assert something about item
assert set(df['item'].unique()) == {'Cheeseburger', 'Foam Finger', 'Rain Poncho'}
# TODO: assert something about category
assert set(df['category'].unique()) == {'Food', 'Merch', 'RainGear'}
print('clean:', df.shape)

clean: (275, 5)


### TODO 7 — the number you would report

**TODO:** add a `revenue` column, then print revenue by category, highest first, plus the overall total. Round money to two decimals.

Then, in one sentence, state what you would tell a vendor to stock more of.

In [200]:
# TODO
df['revenue'] = round((df['qty'] * df['price']), 2)

rev_by_cat = df.groupby('category')['revenue'].sum().round(2).sort_values(ascending = False)
final_rev = rev_by_cat.sum()

print(rev_by_cat)
print('The total revenue is: $', final_rev)




category
Food        1656.0
Merch       1572.0
RainGear    1512.0
Name: revenue, dtype: float64
The total revenue is: $ 4740.0


**What I would tell the vendor:** _Rain Gear is likely only a high seller on rainy days, yet it comes close to food and merch in terms of revenue, so it must be fully stocked when a rainy day does come around._

### TODO 8 — read back your log

In [201]:
import pandas as pd
pd.DataFrame(DECISIONS)

,step,decision,rows
0,Drop duplicates,Copied original data and removed duplicate rows,15
1,Clean price,Removed all dollar signs and changed data type...,300
2,Clean qty,Removed NaN or negative quantity values,25
3,Canonicalize item,Changed all lowercase instances to form with f...,126
4,Normalize category,"Mapped each category to one of 3 categories, t...",132


### Write-up

Two parts.

**a)** Which cleaning step changed your revenue total the most? Give the number before and after that step, not a description.

**b)** Pick one decision you made where a reasonable person could have chosen differently. State the other choice, what it would have done to your reported revenue, and why you went the way you did.

1. The cleaning step that changed the total revenue the most was cleaning the 'qty' column. Prior to that step, the revenue was 4594.50, and after it was 4740.00. It is also worth noting that the revenue could not be calculated with any sort of accuracy until the 'price' column was cleaned into a float, where it could be used in a calculation.
2. One decision I made was to completely drop entries with a NaN or negative quantity value. Considering that these entries were all orders entered into the data frame with values in other columns, it is plausible that there was supposed to be a number > 0 entered. However, I did not want to assign an arbitrary quantity to orders without any quantity, nor make the assumption that a reported negative 1 was supposed to be 1. Those entries also could have been reasonably assumed to be orders that were meant to be canceled, or possibly refunded. I chose to work only with the quantities that made explicit sense in the context of the data frame, without making any assumptions or filling in any gaps.